# Classical Reasoning Agent: Natural-Language KB -> Logic -> True/False

**Pipeline**

`text -> rule-based parser -> Horn clauses (facts + rules, with negation) -> forward chaining (well-founded / stratified negation) -> True/False`

* No generative AI anywhere. The parser is pure pattern matching; the decision is made by our own
  forward-chaining engine (semi-naive, indexed, goal-directed slicing).
* A second engine (backward chaining with unification) is included for cross-checking and for readable traces.
* Negation: explicit negative literals (open-world style) **and** negation-as-failure (closed-world style),
  chosen automatically (`closed_world='auto'`) or forced via config.

In [1]:
import re, time, csv, json, os, sys, itertools
from collections import defaultdict

# ---------------- configuration (edit these) ----------------
CLOSED_WORLD = 'auto'          # 'auto' | True | False   (True = negation as failure)
TEST_PATH    = 'test.csv'      # private test file (csv/json/jsonl) with columns id, context, question
OUT_PATH     = 'submission.csv'
HF_SOURCES = {                 # public datasets used for development (edit ids/configs if needed)
    'RuleTaker': ('tasksource/ruletaker', None),
    'FOLIO':     ('tasksource/folio', None),
}
MAX_EVAL_PER_DATASET = 5000    # cap for quick runs

## 1. Internal representation

* **Term**: constant (`'bob'`, `'bald_eagle'`) or variable (`'?x'`).
* **Atom**: `(pred, args)` where `pred` includes arity, e.g. `'big/1'`, `'chase/2'`.
  An explicitly negated atom uses the predicate `'~big/1'`.
* **Rule**: `head <- pos_body, neg_body` (neg_body = `not p(...)`).
  Unsafe variables (appearing only in negative literals or only in the head) are grounded with a domain predicate `__dom__/1`.

In [2]:
COPULA   = {'is', 'are', 'was', 'were'}
DET      = {'the', 'a', 'an'}
QUANT    = {'all', 'every', 'each', 'any'}
GENERIC  = {'someone', 'something', 'anyone', 'anything', 'everyone', 'everything',
            'people', 'things', 'person', 'thing', 'ones', 'one'}
PRON     = {'they', 'it', 'he', 'she', 'them'}
VERB_STEMS = {'chase', 'eat', 'like', 'need', 'see', 'visit', 'kiss', 'love', 'hate', 'know', 'want',
              'help', 'hit', 'follow', 'meet', 'play', 'watch', 'hug', 'push', 'pull', 'call', 'have',
              'give', 'take', 'make', 'sing', 'teach', 'attack', 'fear', 'trust', 'support', 'visit'}
DOM = '__dom__/1'
SPLIT_CONJ = re.compile(r'\s*,\s*(?:and\s+)?|\s+and\s+', re.I)


def stem_verb(w):
    w = w.lower()
    if w == 'has': return 'have'
    if w.endswith('ies') and len(w) > 4: return w[:-3] + 'y'
    for suf in ('sses', 'shes', 'ches', 'xes', 'zes'):
        if w.endswith(suf): return w[:-2]
    if w.endswith('s') and not w.endswith('ss') and len(w) > 3: return w[:-1]
    return w


def singular(w):
    w = w.lower()
    if w.endswith('ies') and len(w) > 4: return w[:-3] + 'y'
    if w.endswith('ves') and len(w) > 4: return w[:-3] + 'f'
    if w.endswith(('sses', 'shes', 'ches', 'xes')): return w[:-2]
    if w.endswith('s') and not w.endswith(('ss', 'ous', 'us', 'is')) and len(w) > 3: return w[:-1]
    return w


def clean_sentence(s):
    s = s.strip()
    s = re.sub(r"(?i)\b(isn't|aren't|wasn't|weren't)\b", lambda m: m.group(1)[:-3] + ' not', s)
    s = re.sub(r"(?i)\bdoesn't\b", 'does not', s)
    s = re.sub(r"(?i)\bdon't\b", 'do not', s)
    s = re.sub(r"(?i)\bcan't\b", 'can not', s)
    s = re.sub(r'[.!?;:]+$', '', s).strip()
    return re.sub(r'\s+', ' ', s)


def const_term(tokens):
    t = [x.lower() for x in tokens]
    while t and t[0] in DET: t = t[1:]
    return '_'.join(t)


def is_verb_token(t):
    return stem_verb(t) in VERB_STEMS or t.lower() in VERB_STEMS


def find_verb(toks):
    """Return (index, kind) of the main verb; kind in cop|aux|verb. Subject must be non-empty."""
    n = len(toks)
    for i in range(1, n):
        lt = toks[i].lower()
        if lt in COPULA: return i, 'cop'
        if lt in ('does', 'do', 'did') and i + 1 < n and toks[i + 1].lower() == 'not': return i, 'aux'
        if is_verb_token(toks[i]): return i, 'verb'
    for i in range(1, n - 1):                       # heuristic fallback: unknown verb in 3rd person
        lt = toks[i].lower()
        nxt = toks[i + 1]
        if lt.endswith('s') and not lt.endswith('ss') and (nxt.lower() in DET or nxt[0].isupper()):
            return i, 'verb'
    return None, None


def subject_term(toks, pron):
    t = [x.lower() for x in toks]
    if len(t) == 1 and t[0] in PRON:
        return pron if pron is not None else None
    while t and t[0] in QUANT: t = t[1:]
    if len(t) == 1 and t[0] in GENERIC: return '?x'
    return const_term(toks)


def object_term(toks, pron):
    t = [x.lower() for x in toks]
    if len(t) == 1 and t[0] in PRON: return pron
    if len(t) == 1 and t[0] in GENERIC: return '?x'
    return const_term(toks)


def parse_clause(toks, pron=None):
    """Parse one simple clause -> (neg, pred, args) or None."""
    i, kind = find_verb(toks)
    if i is None: return None
    subj = subject_term(toks[:i], pron)
    if subj is None: return None
    if kind == 'cop':
        j = i + 1; neg = False
        if j < len(toks) and toks[j].lower() == 'not': neg = True; j += 1
        rest = [x.lower() for x in toks[j:]]
        while rest and rest[0] in DET: rest = rest[1:]
        if not rest: return None
        return (neg, '_'.join(rest) + '/1', (subj,))
    if kind == 'aux':
        verb, obj_toks, neg = toks[i + 2:i + 3], toks[i + 3:], True
    else:
        verb, obj_toks, neg = toks[i:i + 1], toks[i + 1:], False
    if not verb: return None
    v = stem_verb(verb[0])
    if obj_toks:
        o = object_term(obj_toks, pron)
        if o is None: return None
        return (neg, v + '/2', (subj, o))
    return (neg, v + '/1', (subj,))


def parse_conj(text, pron=None):
    """'x is big and not red, nice' -> list of literals; elliptical conjuncts inherit the subject."""
    lits, prev_subj = [], None
    for part in SPLIT_CONJ.split(text.strip()):
        part = part.strip()
        if not part: continue
        toks = part.split()
        lit = parse_clause(toks, pron)
        if lit is None and prev_subj is not None:
            lit = parse_clause(['__s__'] + toks, prev_subj if prev_subj else pron)
            if lit is None:
                neg = toks[0].lower() == 'not'
                rest = [x.lower() for x in (toks[1:] if neg else toks)]
                while rest and rest[0] in DET: rest = rest[1:]
                if rest: lit = (neg, '_'.join(rest) + '/1', (prev_subj,))
        if lit is None: return None
        lits.append(lit)
        prev_subj = lit[2][0]
    return lits

## 2. Knowledge base construction (sentences -> facts + rules)

In [3]:
class Rule:
    __slots__ = ('head', 'pos', 'neg', 'text')
    def __init__(self, head, pos, neg):
        self.head, self.pos, self.neg = head, pos, neg
        self.text = self.pretty()
    def pretty(self):
        f = lambda a: f"{a[0]}({','.join(a[1])})"
        b = [f(a) for a in self.pos if a[0] != DOM] + ['not ' + f(a) for a in self.neg]
        return ('IF ' + ' AND '.join(b) + ' THEN ' if b else 'ALWAYS ') + f(self.head)


def vars_of(atom):
    return {a for a in atom[1] if a.startswith('?')}


def make_rule(head_lit, body_lits):
    hneg, hp, ha = head_lit
    head = (('~' if hneg else '') + hp, ha)
    pos = [(p, a) for n, p, a in body_lits if not n]
    neg = [(p, a) for n, p, a in body_lits if n]
    bound = set().union(*[vars_of(x) for x in pos]) if pos else set()
    need = (vars_of(head) | set().union(*[vars_of(x) for x in neg]) if neg else vars_of(head)) - bound
    for v in sorted(need): pos.append((DOM, (v,)))
    return Rule(head, pos, neg)


class KB:
    def __init__(self):
        self.facts = defaultdict(set)      # pred -> set(args tuples)
        self.rules = []
        self.consts = set()
        self.explicit_neg = False
        self.unparsed = []

    def add_literal_fact(self, lit):
        neg, p, a = lit
        self.explicit_neg |= neg
        self.facts[('~' if neg else '') + p].add(a)
        self.consts.update(a)

    def add_rule(self, r):
        self.explicit_neg |= r.head[0].startswith('~')
        self.rules.append(r)
        for _, a in r.pos + r.neg:
            self.consts.update(x for x in a if not x.startswith('?'))
        self.consts.update(x for x in r.head[1] if not x.startswith('?'))


def split_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+|\n+', text.strip())
    return [p for p in (x.strip() for x in parts) if p]


def add_sentence(kb, sent):
    s = clean_sentence(sent)
    toks = s.split()
    if not toks: return True
    low = [t.lower() for t in toks]
    # ---- conditional: "If B1 and B2 then H"
    if low[0] == 'if':
        halves = re.split(r',?\s+then\s+', s[3:], maxsplit=1, flags=re.I)
        if len(halves) != 2: return False
        body = parse_conj(halves[0])
        if not body: return False
        head = parse_conj(halves[1], pron=body[0][2][0])
        if not head: return False
        for h in head: kb.add_rule(make_rule(h, body))
        return True
    # ---- universal / quantified statements ("Big, red things are nice", "All dogs are mammals", "No cats bark")
    i, kind = find_verb(toks)
    if i is None: return False
    subj_toks = toks[:i]
    q = subj_toks[0].lower()
    body, rest_subj = [], subj_toks
    quant_no = (q == 'no')
    if q in QUANT or q == 'no':
        rest_subj = subj_toks[1:]
    if (q in QUANT or quant_no or subj_toks[-1].lower() in GENERIC) and rest_subj:
        # noun-like head word: generic words add no predicate; real plural nouns do
        last = rest_subj[-1].lower()
        adj_part = rest_subj[:-1] if last in GENERIC else rest_subj[:-1]
        text_adj = ' '.join(adj_part)
        if last not in GENERIC:
            body.append((False, singular(last) + '/1', ('?x',)))
        for a in [x for x in SPLIT_CONJ.split(text_adj) if x.strip()]:
            at = a.split(); n = at[0].lower() == 'not'
            at = [w.lower() for w in (at[1:] if n else at)]
            if at: body.append((n, '_'.join(at) + '/1', ('?x',)))
        head_toks = toks[i:]
        if kind == 'cop' and last not in GENERIC:          # "All dogs are mammals": singularise the predicate noun
            head_toks = head_toks[:-1] + [singular(head_toks[-1])]
        head_txt = 'they ' + ' '.join(head_toks)
        head = parse_conj(head_txt, pron='?x')
        if not head: return False
        for h in head:
            if quant_no: h = (not h[0], h[1], h[2])
            kb.add_rule(make_rule(h, body))
        return True
    # ---- ground fact(s)
    lits = parse_conj(s)
    if not lits: return False
    for l in lits:
        if any(a.startswith('?') for a in l[2]): kb.add_rule(make_rule(l, []))
        else: kb.add_literal_fact(l)
    return True


def build_kb(context):
    kb = KB()
    for sent in split_sentences(context):
        try: ok = add_sentence(kb, sent)
        except Exception: ok = False
        if not ok: kb.unparsed.append(sent)
    return kb


def parse_query(q):
    q = re.sub(r'(?i)^\s*(query|statement|question|hypothesis)\s*:\s*', '', q)
    q = split_sentences(q)[0] if split_sentences(q) else q
    s = clean_sentence(q)
    lits = parse_conj(s)
    return lits[0] if lits else None

## 3. Forward chaining engine

Semi-naive evaluation over a predicate index. Negation is handled with the **alternating fixpoint**
(`T_{k+1} = Gamma(Gamma(T_k))`), which equals the stratified model on stratified programs and the
well-founded model otherwise. If no rule has a negative body literal, a single pass is enough.
Before reasoning, rules/facts irrelevant to the query predicate are pruned (backward dependency slicing).

In [4]:
def slice_kb(kb, q_pred):
    base = lambda p: p.lstrip('~')
    need, stack = {base(q_pred)}, [base(q_pred)]
    by_head = defaultdict(list)
    for r in kb.rules: by_head[base(r.head[0])].append(r)
    while stack:
        p = stack.pop()
        for r in by_head.get(p, ()):
            for bp, _ in r.pos + r.neg:
                b = base(bp)
                if b not in need and b != DOM:
                    need.add(b); stack.append(b)
    rules = [r for r in kb.rules if base(r.head[0]) in need]
    facts = {p: s for p, s in kb.facts.items() if base(p) in need}
    return rules, facts


def match(pattern, fact, sub):
    new = None
    for p, f in zip(pattern, fact):
        if p[0] == '?':
            cur = (new or sub).get(p)
            if cur is None:
                if new is None: new = dict(sub)
                new[p] = f
            elif cur != f: return None
        elif p != f: return None
    return new if new is not None else sub


def subst(args, sub):
    return tuple(sub.get(a, a) for a in args)


def gamma(rules, facts, consts, ref, cwa, with_why=False):
    """Least model of the program in which `not p` holds iff explicit ~p is derived or (cwa and p not in ref)."""
    known = defaultdict(set)
    for p, s in facts.items(): known[p] |= s
    known[DOM] = {(c,) for c in consts}
    delta = defaultdict(set)
    for p, s in known.items(): delta[p] = set(s)
    why = {}
    while delta:
        new = defaultdict(set)
        for r in rules:
            npos = len(r.pos)
            for k in range(npos):
                if not delta.get(r.pos[k][0]): continue
                # literal k joins against delta, all others against everything known
                def rec(idx, sub, support):
                    if idx == npos:
                        for pn, an in r.neg:
                            g = subst(an, sub)
                            if g in known.get('~' + pn, ()): continue          # explicit negation holds
                            if cwa and g not in ref.get(pn, ()): continue      # negation as failure holds
                            return
                        h = (r.head[0], subst(r.head[1], sub))
                        if h[1] not in known[h[0]] and h[1] not in new[h[0]]:
                            new[h[0]].add(h[1])
                            if with_why: why[h] = (r.text, list(support), [(pn, subst(an, sub)) for pn, an in r.neg])
                        return
                    pn, an = r.pos[idx]
                    src = delta.get(pn, ()) if idx == k else known.get(pn, ())
                    for f in src:
                        s2 = match(an, f, sub)
                        if s2 is not None:
                            rec(idx + 1, s2, support + [(pn, f)] if pn != DOM else support)
                rec(0, {}, [])
        delta = defaultdict(set)
        for p, s in new.items():
            s -= known[p]
            if s: known[p] |= s; delta[p] = s
    return known, why


class ForwardChainer:
    def __init__(self, kb, closed_world='auto'):
        self.kb = kb
        self.cwa = (not kb.explicit_neg) if closed_world == 'auto' else bool(closed_world)

    def entails(self, query, with_why=False):
        neg, p, a = query
        rules, facts = slice_kb(self.kb, p)
        consts = self.kb.consts | set(a)
        has_neg_body = any(r.neg for r in rules)
        if not has_neg_body or not self.cwa:
            T, why = gamma(rules, facts, consts, {}, self.cwa, with_why)
            over = T
        else:
            T = {}
            while True:
                over, _ = gamma(rules, facts, consts, T, self.cwa)
                newT, why = gamma(rules, facts, consts, over, self.cwa, with_why)
                if {k: v for k, v in newT.items() if v} == {k: v for k, v in T.items() if v}: break
                T = newT
            T = newT
        self.T, self.why = T, why
        if not neg: ans = a in T.get(p, ())
        else: ans = a in T.get('~' + p, ()) or (self.cwa and a not in over.get(p, ()))
        return ans

    # ---- reasoning trace
    def explain(self, query, indent=0, seen=None):
        neg, p, a = query
        seen = seen if seen is not None else set()
        fmt = lambda pp, aa: f"{'NOT ' if pp.startswith('~') else ''}{pp.lstrip('~').split('/')[0]}({', '.join(aa)})"
        pad = '  ' * indent
        key = (('~' if neg else '') + p, a)
        if neg and not (a in self.T.get('~' + p, ())):
            return pad + f"{fmt('~' + p, a)}  [negation as failure: {fmt(p, a)} cannot be derived]\n"
        if key in seen: return pad + f"{fmt(*key)}  [see above]\n"
        seen.add(key)
        if a in self.kb.facts.get(key[0], ()): return pad + f"{fmt(*key)}  [given fact]\n"
        w = self.why.get(key)
        if w is None: return pad + f"{fmt(*key)}  [no derivation]\n"
        out = pad + f"{fmt(*key)}  <= {w[0]}\n"
        for pn, f in w[1]:
            out += self.explain((pn.startswith('~'), pn.lstrip('~'), f), indent + 1, seen)
        for pn, f in w[2]:
            out += pad + f"  not {fmt(pn, f)}  [holds: no derivation of {fmt(pn, f)}]\n"
        return out

## 4. Backward chaining with unification (verification engine)

Goal-driven SLD resolution with loop checking. Rule heads are unified with the goal; remaining
variables are enumerated over the domain. Used to cross-check the forward chainer.

In [5]:
class BackwardChainer:
    def __init__(self, kb, closed_world='auto', max_calls=200000):
        self.kb, self.max_calls = kb, max_calls
        self.cwa = (not kb.explicit_neg) if closed_world == 'auto' else bool(closed_world)
        self.by_head = defaultdict(list)
        for r in kb.rules: self.by_head[r.head[0]].append(r)

    def prove(self, goal, anc):
        self.calls += 1
        if self.calls > self.max_calls: raise RuntimeError('budget')
        p, a = goal
        if a in self.kb.facts.get(p, ()): return True
        if goal in anc: return False
        anc = anc | {goal}
        for r in self.by_head.get(p, ()):
            sub = {}
            ok = True
            for h, g in zip(r.head[1], a):                      # unify head with ground goal
                if h[0] == '?':
                    if sub.setdefault(h, g) != g: ok = False; break
                elif h != g: ok = False; break
            if ok and self.solve(r, 0, sub, anc): return True
        return False

    def solve(self, r, idx, sub, anc):
        if idx == len(r.pos):
            for pn, an in r.neg:
                g = subst(an, sub)
                if self.prove(('~' + pn, g), anc): continue
                if self.cwa and not self.prove((pn, g), anc): continue
                return False
            return True
        pn, an = r.pos[idx]
        free = [x for x in dict.fromkeys(an) if x[0] == '?' and x not in sub]
        for combo in itertools.product(sorted(self.kb.consts), repeat=len(free)):
            s2 = dict(sub); s2.update(zip(free, combo))
            g = subst(an, s2)
            if pn == DOM or self.prove((pn, g), anc):
                if self.solve(r, idx + 1, s2, anc): return True
        return False

    def entails(self, query):
        neg, p, a = query
        self.calls = 0
        self.kb.consts |= set(a)
        if not neg: return self.prove((p, a), frozenset())
        return self.prove(('~' + p, a), frozenset()) or (self.cwa and not self.prove((p, a), frozenset()))

## 5. Prediction API

In [6]:
def predict(context, question, closed_world=CLOSED_WORLD, trace=False):
    kb = build_kb(context)
    q = parse_query(question)
    if q is None: return (False, '') if trace else False
    fc = ForwardChainer(kb, closed_world)
    ans = fc.entails(q, with_why=trace)
    return (ans, fc.explain(q) if (trace and ans and not q[0]) else '') if trace else ans

## 6. Sanity tests (RuleTaker-style, hand written)

In [7]:
TESTS = [
 ("Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is successful.", True),
 ("Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is not successful.", False),
 ("The cat is big. The cat is not red. If something is big and not red then it is rough. Rough things are kind.", "The cat is kind.", True),
 ("The bald eagle chases the dog. The dog is big. If something chases the dog then it is red. Red things are nice.", "The bald eagle is nice.", True),
 ("The bald eagle chases the dog. The dog is big. If something chases the dog then it is red. Red things are nice.", "The dog is nice.", False),
 ("Anne is big. Anne is red. If something is big and red then it is not nice. All nice things are young.", "Anne is not nice.", True),
 ("Dave is cold. If someone is not cold then they are happy. If someone is happy then they like the cat.", "Dave likes the cat.", False),
 ("Eve is quiet. If someone is not cold then they are happy. If someone is happy then they like the cat.", "Eve likes the cat.", True),
 ("The tiger likes the cow. The cow needs the tiger. If someone likes the cow and the cow needs the tiger then the cow sees the tiger.", "The cow sees the tiger.", True),
 ("Big, red things are nice. Fred is big. Fred is red. Gina is big.", "Gina is nice.", False),
 ("Big, red things are nice. Fred is big. Fred is red. Gina is big.", "Fred is nice.", True),
 ("All dogs are mammals. Rex is a dog. No mammals are fish.", "Rex is a mammal.", True),
 ("All dogs are mammals. Rex is a dog. No mammals are fish.", "Rex is not a fish.", True),
 ("Harry is nice. Harry is rough and kind. If something is rough and kind then it is not red.", "Harry is red.", False),
 ("If the bear is red then the bear is kind. The bear is red.", "The bear is kind.", True),
 ("If someone is red, then they are round. Ann is red. Everything is big.", "Ann is round.", True),
 ("The lion does not like the bear. If someone does not like the bear then they eat the rabbit.", "The lion eats the rabbit.", True),
]

def run_tests():
    ok = 0
    for ctx, q, gold in TESTS:
        kb = build_kb(ctx); ql = parse_query(q)
        pf = ForwardChainer(kb).entails(ql) if ql else False
        try: pb = BackwardChainer(build_kb(ctx)).entails(ql) if ql else False
        except RuntimeError: pb = None
        good = (pf == gold) and (pb == pf)
        ok += good
        if not good: print('FAIL', q, 'gold', gold, 'fwd', pf, 'bwd', pb, kb.unparsed)
    print(f'{ok}/{len(TESTS)} sanity tests passed (forward == backward == gold)')
run_tests()

17/17 sanity tests passed (forward == backward == gold)


## 7. Data loading (Hugging Face with local fallback)

In [8]:
CTX_COLS = ['context', 'theory', 'premises', 'premise', 'text', 'story', 'knowledge_base']
Q_COLS   = ['question', 'query', 'statement', 'hypothesis', 'conclusion', 'sentence']
LBL_COLS = ['label', 'answer', 'gold', 'label_text', 'flag', 'prediction']

def to_bool(v):
    s = str(v).strip().lower()
    if s in ('true', 'entailment', '1', 'yes', 'entailed'): return True
    return False                      # False / not entailment / unknown / uncertain / 0 / no

def pick(row, cols):
    for c in cols:
        if c in row and row[c] is not None: return row[c]
    return None

def normalise_rows(rows, name):
    out = []
    for i, r in enumerate(rows):
        ctx, q, l = pick(r, CTX_COLS), pick(r, Q_COLS), pick(r, LBL_COLS)
        if isinstance(ctx, list): ctx = ' '.join(ctx)
        if ctx is None or q is None: continue
        out.append({'id': r.get('id', f'{name}-{i}'), 'context': ctx, 'question': q,
                    'label': to_bool(l) if l is not None else None})
    return out

def load_hf(name, hf_id, cfg):
    from datasets import load_dataset
    ds = load_dataset(hf_id, cfg) if cfg else load_dataset(hf_id)
    split = 'test' if 'test' in ds else ('validation' if 'validation' in ds else list(ds.keys())[0])
    return normalise_rows(list(ds[split]), name)

def load_file(path):
    if path.endswith('.csv'):
        import pandas as pd
        return pd.read_csv(path).to_dict('records')
    if path.endswith('.jsonl'):
        return [json.loads(l) for l in open(path) if l.strip()]
    d = json.load(open(path))
    return d if isinstance(d, list) else list(d.values())

DEMO = [{'id': f'demo-{i}', 'context': c, 'question': q, 'label': g} for i, (c, q, g) in enumerate(TESTS)]

def get_public_sets():
    sets = {}
    for name, (hf_id, cfg) in HF_SOURCES.items():
        try:
            sets[name] = load_hf(name, hf_id, cfg)[:MAX_EVAL_PER_DATASET]
            print(f'loaded {name}: {len(sets[name])} rows')
        except Exception as e:
            print(f'[{name}] could not load from Hugging Face ({type(e).__name__}); looking for local data/{name}.*')
            for ext in ('jsonl', 'json', 'csv'):
                p = f'data/{name}.{ext}'
                if os.path.exists(p):
                    sets[name] = normalise_rows(load_file(p), name)[:MAX_EVAL_PER_DATASET]; break
    sets = {k: v for k, v in sets.items() if v}
    if not sets:
        print('No public data available -> evaluating on the built-in demo set only.')
        sets['demo'] = DEMO
    return sets

## 8. Evaluation: accuracy and inference time

In [9]:
def evaluate(rows, closed_world=CLOSED_WORLD):
    t0 = time.perf_counter(); correct = 0; n = 0
    for r in rows:
        pred = predict(r['context'], r['question'], closed_world)
        if r['label'] is not None:
            n += 1; correct += (pred == r['label'])
    dt = time.perf_counter() - t0
    return correct / max(n, 1) * 100, dt, n

public = get_public_sets()
RESULTS = {}
for name, rows in public.items():
    acc, dt, n = evaluate(rows)
    RESULTS[name] = (acc, dt, n, len(rows))
    print(f'{name:10s} accuracy = {acc:6.2f}%   queries = {n}   total inference time = {dt:.3f}s  ({dt / max(len(rows), 1) * 1000:.2f} ms/query)')

[RuleTaker] could not load from Hugging Face (ModuleNotFoundError); looking for local data/RuleTaker.*
[FOLIO] could not load from Hugging Face (ModuleNotFoundError); looking for local data/FOLIO.*
No public data available -> evaluating on the built-in demo set only.
demo       accuracy = 100.00%   queries = 17   total inference time = 0.003s  (0.19 ms/query)


## 9. Generate `submission.csv` for the private test set

In [10]:
TEST_INFO = {}

def make_submission(test_path=TEST_PATH, out_path=OUT_PATH):
    rows = normalise_rows(load_file(test_path), 'test')
    t0 = time.perf_counter()
    preds = [predict(r['context'], r['question']) for r in rows]
    dt = time.perf_counter() - t0
    with open(out_path, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['id', 'label'])
        for r, p in zip(rows, preds): w.writerow([r['id'], 'True' if p else 'False'])
    print(f'wrote {out_path}: {len(rows)} predictions in {dt:.3f}s')
    TEST_INFO['time'], TEST_INFO['n'] = dt, len(rows)
    if any(r['label'] is not None for r in rows):
        TEST_INFO['acc'] = sum(p == r['label'] for p, r in zip(preds, rows)) / len(rows) * 100
        print(f"accuracy on supplied labels: {TEST_INFO['acc']:.2f}%")

if os.path.exists(TEST_PATH):
    make_submission()
else:
    print(f'{TEST_PATH} not found - writing a placeholder submission from the demo set instead.')
    t0 = time.perf_counter()
    demo_preds = [predict(r['context'], r['question']) for r in DEMO]
    TEST_INFO['time'], TEST_INFO['n'] = time.perf_counter() - t0, len(DEMO)
    with open(OUT_PATH, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['id', 'label'])
        for r, p in zip(DEMO, demo_preds): w.writerow([r['id'], 'True' if p else 'False'])

test.csv not found - writing a placeholder submission from the demo set instead.


## 10. Final results (accuracy and inference time)

In [11]:
tot_n = sum(v[2] for v in RESULTS.values())
tot_c = sum(v[0] / 100 * v[2] for v in RESULTS.values())
tot_t = sum(v[1] for v in RESULTS.values())
print('=' * 62)
print('FINAL RESULTS')
print('=' * 62)
for name, (acc, dt, n, total) in RESULTS.items():
    print(f'{name:12s} accuracy = {acc:6.2f}%   queries = {n:5d}   inference time = {dt:.3f}s')
print('-' * 62)
print(f'OVERALL ACCURACY (labelled public data): {tot_c / max(tot_n, 1) * 100:.2f}%  over {tot_n} queries')
print(f'TOTAL INFERENCE TIME (public data)     : {tot_t:.3f}s')
if TEST_INFO:
    print(f"TEST/SUBMISSION INFERENCE TIME         : {TEST_INFO['time']:.3f}s for {TEST_INFO['n']} queries -> {OUT_PATH}")
    if 'acc' in TEST_INFO: print(f"TEST ACCURACY (if labels supplied)     : {TEST_INFO['acc']:.2f}%")
print('=' * 62)

FINAL RESULTS
demo         accuracy = 100.00%   queries =    17   inference time = 0.003s
--------------------------------------------------------------
OVERALL ACCURACY (labelled public data): 100.00%  over 17 queries
TOTAL INFERENCE TIME (public data)     : 0.003s
TEST/SUBMISSION INFERENCE TIME         : 0.003s for 17 queries -> submission.csv


## 11. Example reasoning traces

In [12]:
for ctx, q, _ in [TESTS[0], TESTS[3], TESTS[7]]:
    ans, tr = predict(ctx, q, trace=True)
    print('KB   :', ctx); print('QUERY:', q); print('ANSWER:', ans); print(tr or '(negative / failed query: no positive derivation)'); print('-' * 60)

KB   : Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.
QUERY: Bob is successful.
ANSWER: True
successful(bob)  <= IF smart/1(?x) AND kind/1(?x) THEN successful/1(?x)
  smart(bob)  <= IF young/1(?x) THEN smart/1(?x)
    young(bob)  [given fact]
  kind(bob)  [given fact]

------------------------------------------------------------
KB   : The bald eagle chases the dog. The dog is big. If something chases the dog then it is red. Red things are nice.
QUERY: The bald eagle is nice.
ANSWER: True
nice(bald_eagle)  <= IF red/1(?x) THEN nice/1(?x)
  red(bald_eagle)  <= IF chase/2(?x,dog) THEN red/1(?x)
    chase(bald_eagle, dog)  [given fact]

------------------------------------------------------------
KB   : Eve is quiet. If someone is not cold then they are happy. If someone is happy then they like the cat.
QUERY: Eve likes the cat.
ANSWER: True
like(eve, cat)  <= IF happy/1(?x) THEN like/2(?x,cat)
  happy(eve)  <= 